# Spark-X2.6 — LSA-F2: faithful D-LSA recall gate on REAL attention (doc `07 §3`)

Distils per-layer index maps (per-key KL to the frozen model's attention) on long documents, then measures **attention-mass recall at k=2048** on **held-out** documents for: `faithful` (learned space + Ward buckets), `flat_oracle_index` (same scores, no clustering), `contiguous` (the repo's previous selector), `random`, `oracle`.

* Gate (claim-grade only if val length >= 64K): faithful >= 0.90, >= contiguous + 0.10, flat_oracle - faithful <= 0.03. Layers that fail keep the exact path.
* GPU path is **untested**; errors are collected in `REPORT['errors']` instead of raising. Needs ~8 GB for the model + ~3-4 GB per layer captured at 64K (`LAYERS_PER_PASS`).
* Cells: 1 sync - 2 load model - 3 data - 4 run LSA-F2 - 5 report (copy-paste JSON) - 6 install indexers for decode experiments


In [ ]:
# ==============================================================================
# Cell 1: Clone / Sync Repository & Install Dependencies
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

pip_pkgs = [
    'torch>=2.5.0',
    'triton>=3.1.0',
    'transformers==4.57.1',
    'datasets>=3.2.0',
    'accelerate>=1.2.0',
    'safetensors>=0.4.5',
    'huggingface_hub>=0.27.0',
    'bitsandbytes>=0.45.0',
    'cut-cross-entropy',
    'liger-kernel',
    'unsloth',
    'openai>=1.58.0',
    'nltk',
    'langdetect',
    'psutil',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_pkgs], check=False)
print('Working directory synced:', Path.cwd())

In [ ]:
# Cell 2: load model (no training config needed)
import os, importlib, subprocess, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_dlsa, sparkx_lsa_eval
for _m in (sparkx_dlsa, sparkx_lsa_eval):
    importlib.reload(_m)
from load_inference_v03 import load_sparkx_loop_v03
def _get_secret(name):
    v = os.environ.get(name)
    if v: return v.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v.strip()
    except Exception:
        pass
    return None
HF_TOKEN = _get_secret('HF_TOKEN')
HF_REPO_ID = 'TassP0lar/Spark-X2.6-4B-En4B-A1B'     # or the long-run `best` checkpoint once available
model, tokenizer = load_sparkx_loop_v03(repo_id=HF_REPO_ID, device='cuda' if torch.cuda.is_available() else 'cpu',
                                        token=HF_TOKEN, enable_loglinear_kv=True, use_pristine_base=False)
model.eval()
print('model ready; free GPU GB:', round(torch.cuda.mem_get_info()[0] / 1e9, 1) if torch.cuda.is_available() else 'cpu')


In [ ]:
# Cell 3: long documents (train/val are DISJOINT sources). Prefers real long documents (PG-19 books); falls back to packed web text.
import itertools
from datasets import load_dataset
N_TRAIN_DOCS, N_VAL_DOCS = 40, 12
def _take(ds, n, max_chars=1_500_000):
    return [x['text'][:max_chars] for x in itertools.islice(ds, n) if x.get('text')]
try:
    train_texts = _take(load_dataset('emozilla/pg19', split='train', streaming=True), N_TRAIN_DOCS)
    val_texts   = _take(load_dataset('emozilla/pg19', split='validation', streaming=True), N_VAL_DOCS)
    SOURCE = 'pg19 books (train vs validation split)'
except Exception as e:
    print('pg19 unavailable:', repr(e)[:150], '-> fallback to packed fineweb-edu')
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    allt = _take(ds, 6000, 20000)
    train_texts, val_texts = allt[:5000], allt[5000:]
    SOURCE = 'fineweb-edu packed (weak long-range structure: weaker test!)'
print(SOURCE, '| train docs', len(train_texts), '| val docs', len(val_texts))


In [ ]:
# Cell 4: run LSA-F2 (about 10-40 min on A100 depending on lengths)
import sparkx_lsa_eval as le, json
TRAIN_LENGTHS = (16384, 32768)
VAL_LENGTHS   = (32768, 65536)       # >= 65536 is required for a claim-grade gate
LAYERS_PER_PASS = 2                  # lower to 1 on <= 24 GB GPUs
REPORT = le.run_lsa_f2(model, tokenizer, train_texts, val_texts, layers=le.GLOBAL_LAYERS,
                       train_lengths=TRAIN_LENGTHS, val_lengths=VAL_LENGTHS, train_docs_per_length=3, val_docs_per_length=3,
                       layers_per_pass=LAYERS_PER_PASS, k_budget=2048, distill_steps=300, out_dir='/content/lsa_f2')
print('errors:', REPORT['errors'][:10], '| seconds:', REPORT.get('seconds'))


In [ ]:
# Cell 5: report + gate (paste the JSON back)
print(le.format_table(REPORT))
GATE = le.gate_f2(REPORT)
print('\nGATE: claim_grade=%s all_pass=%s passed=%s use_exact=%s' % (GATE.get('claim_grade'), GATE.get('all_pass'),
      GATE.get('layers_passed'), GATE.get('layers_failed_use_exact_path')))
print('\n=== COPY-PASTE LSA-F2 JSON ===')
print(json.dumps({'source': SOURCE, 'gate': GATE, 'config': REPORT['config'], 'errors': REPORT['errors'][:10],
                  'kl': {l: v.get('distill') for l, v in REPORT['layers'].items()},
                  'fidelity_and_layout': {l: v.get('rope_layout') for l, v in REPORT['layers'].items()}}, indent=1))


In [ ]:
# Cell 6 (OPTIONAL): install distilled indexers into the model for decode experiments (LSA-F3), only for layers that passed
passed = set(GATE.get('layers_passed') or [])
inds = {l: d for l, d in le.load_indexers('/content/lsa_f2/dlsa_indexers.pt').items() if l in passed}
print('upgraded layers:', le.upgrade_lsa_indexers(model, inds, k_budget=2048))
